# Exact binary-memory lifetime and correction heat
Upload the standalone manuscript ZIP. This runs the verification and regenerates the figures and CSVs using manual upload. It does not compile LaTeX.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, runpy, subprocess, sys, contextlib, io, shutil
from IPython.display import display, Image
uploaded=files.upload()
if len(uploaded)!=1:
    raise ValueError("Upload the standalone manuscript ZIP only")
name=next(iter(uploaded))
project=Path("memory_lifetime_reproduction")
(project/"supplementary").mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(name) as archive:
    for member in ("supplementary/verify_memory.py", "supplementary/make_figures.py", "supplementary/review_checks.py", "requirements-tested.txt", "supplementary/environment.json"):
        (project/member).write_bytes(archive.read(member))
print("Recorded tested environment:", (project/"supplementary/environment.json").read_text())
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(project/"requirements-tested.txt")])
for script,output in [("verify_memory.py","verification_output.txt"),("make_figures.py","figure_generation_output.txt"),("review_checks.py","review_checks_output.txt")]:
    buffer=io.StringIO()
    with contextlib.redirect_stdout(buffer):
        runpy.run_path(str(project/"supplementary"/script),run_name="__main__")
    (project/"supplementary"/output).write_text(buffer.getvalue())
    print(buffer.getvalue())
for figure in ("retention_bounds.png","optimal_heat.png"):
    display(Image(filename=str(project/"figures"/figure)))
result=shutil.make_archive("memory_lifetime_reproduction_results","zip",root_dir=project)
files.download(result)
